In [1]:
import os
import re
import json
import sqlite3
import requests
from uuid import uuid4
from datetime import datetime

In [2]:
try:
    import pdfplumber
    HAS_PDF = True
except:
    HAS_PDF = False

try:
    from docx import Document
    HAS_DOCX = True
except:
    HAS_DOCX = False

In [3]:
INPUT_FOLDER = "candidate_profile"
DB_PATH = "candidates.db"
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3" 

In [4]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS candidates (
    id TEXT PRIMARY KEY,
    filename TEXT,
    name TEXT,
    skills TEXT,
    experience_years REAL,
    roles TEXT,
    education TEXT,
    summary TEXT,
    processed_at TEXT
)
""")
conn.commit()

In [5]:
def is_processed(filename):
    cursor.execute("SELECT 1 FROM candidates WHERE filename=?", (filename,))
    return cursor.fetchone() is not None


In [6]:
def extract_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".pdf" and HAS_PDF:
        with pdfplumber.open(filepath) as pdf:
            return "\n".join(p.extract_text() or "" for p in pdf.pages)

    elif ext in [".docx", ".doc"] and HAS_DOCX:
        doc = Document(filepath)
        return "\n".join(p.text for p in doc.paragraphs)

    else:
        with open(filepath, "r", errors="ignore") as f:
            return f.read()

In [7]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))

    return chunks

In [9]:
def build_prompt(resume_text):
    return f"""
Extract structured information in JSON format with keys:
name, skills, experience_years, roles, education, summary

Resume:
<<<
{resume_text}
>>>
"""

In [10]:
def call_ollama(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2}
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=30)
    response.raise_for_status()

    return response.json().get("response", "")

In [11]:
def parse_json(text):
    text = re.sub(r"```json|```", "", text).strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group())

    return json.loads(text)

In [12]:
def validate_output(data):
    return {
        "name": data.get("name", "Unknown"),
        "skills": list(set([s.lower() for s in data.get("skills", [])])),
        "experience_years": float(data.get("experience_years", 0)),
        "roles": data.get("roles", []),
        "education": data.get("education", []),
        "summary": data.get("summary", "")[:500]
    }


In [1]:
def save_to_db(filename, data):
    cursor.execute("""
    INSERT INTO candidates VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        str(uuid4()),
        filename,
        data["name"],
        json.dumps(data["skills"]),
        data["experience_years"],
        json.dumps(data["roles"]),
        json.dumps(data["education"]),
        data["summary"],
    ))
    conn.commit()

In [ ]:
def process_candidates():
    for file in os.listdir(INPUT_FOLDER):
        path = os.path.join(INPUT_FOLDER, file)

        if not os.path.isfile(path):
            continue

        if is_processed(file):
            print(f"Skipped: {file}")
            continue

        print(f"Processing: {file}")

        try:
            text = extract_text(path)

            if not text.strip():
                print(f"Empty file: {file}")
                continue

            cleaned = clean_text(text)

            
            chunks = chunk_text(cleaned, chunk_size=500, overlap=50)

            
            combined_data = {
                "name": "",
                "skills": [],
                "experience_years": 0,
                "roles": [],
                "education": [],
                "summary": ""
            }

            for chunk in chunks:
                prompt = build_prompt(chunk)

                for attempt in range(2):
                    try:
                        raw = call_ollama(prompt)
                        parsed = parse_json(raw)
                        validated = validate_output(parsed)

                        # 🔹 NEW: Merge logic
                        if not combined_data["name"]:
                            combined_data["name"] = validated["name"]

                        combined_data["skills"].extend(validated["skills"])
                        combined_data["roles"].extend(validated["roles"])
                        combined_data["education"].extend(validated["education"])

                        combined_data["experience_years"] = max(
                            combined_data["experience_years"],
                            validated["experience_years"]
                        )

                        combined_data["summary"] += " " + validated["summary"]

                        break
                    except Exception:
                        if attempt == 1:
                            continue

            combined_data["skills"] = list(set(combined_data["skills"]))
            combined_data["roles"] = list(set(combined_data["roles"]))
            combined_data["education"] = list(set(combined_data["education"]))
            combined_data["summary"] = combined_data["summary"][:500]

            save_to_db(file, combined_data)
            print(f"Saved: {file}")

        except Exception as e:
            print(f"Error processing {file}: {e}")

In [15]:
if __name__ == "__main__":
    process_candidates()

Skipped: Bharathi DevOps Engineer.pdf
Skipped: EVLYN MONICA JOSEPH.pdf
Skipped: Pravin _Resume.pdf
Skipped: Resume-latest.pdf
Processing: VRR 2026 (1).pdf
Saved: VRR 2026 (1).pdf


C:\Users\anupt\AppData\Local\Temp\ipykernel_30852\3133401936.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat()
